In [1]:
# 05_evaluation_experiments.ipynb

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Load processed features
# --------------------------------------------------
BASE_DIR = Path.cwd().parent
TRAIN_CSV = BASE_DIR / "data" / "processed" / "train_features_final.csv"

df = pd.read_csv(TRAIN_CSV)
target_col = "TARGET"

X = df.drop(columns=[target_col])
y = df[target_col]

# --------------------------------------------------
# Train/Validation split for experiments
# --------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --------------------------------------------------
# Models to experiment with
# --------------------------------------------------
models = {
    "logistic_regression": LogisticRegression(max_iter=500, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "xgboost": XGBClassifier(
        n_estimators=100, use_label_encoder=False, eval_metric="logloss", random_state=42
    )
}

# --------------------------------------------------
# Define evaluation metrics functions
# --------------------------------------------------
def ks_statistic(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return max(tpr - fpr)

def gini_coefficient(y_true, y_prob):
    return 2 * roc_auc_score(y_true, y_prob) - 1

def calibration_error(y_true, y_prob, n_bins=10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    return np.mean(np.abs(prob_true - prob_pred))

# --------------------------------------------------
# Train models and compute metrics
# --------------------------------------------------
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        y_val_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_val_prob = model.predict(X_val)

    metrics = {
        "ks": ks_statistic(y_val, y_val_prob),
        "gini": gini_coefficient(y_val, y_val_prob),
        "auc": roc_auc_score(y_val, y_val_prob),
        "brier": brier_score_loss(y_val, y_val_prob),
        "calibration_error": calibration_error(y_val, y_val_prob)
    }

    results[name] = metrics
    print(f"{name} metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

# --------------------------------------------------
# Identify best model by KS
# --------------------------------------------------
champion_name = max(results, key=lambda k: results[k]["ks"])
print(f"\nChampion model (by KS): {champion_name}")



Training logistic_regression...
logistic_regression metrics:
  ks: 0.3372
  gini: 0.4506
  auc: 0.7253
  brier: 0.0700
  calibration_error: 0.0666

Training random_forest...
random_forest metrics:
  ks: 0.2481
  gini: 0.3364
  auc: 0.6682
  brier: 0.0725
  calibration_error: 0.1287

Training xgboost...


c:\Users\Admin\Desktop\CREDIT_RISK_SCORING\env2\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:18:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgboost metrics:
  ks: 0.3011
  gini: 0.4039
  auc: 0.7019
  brier: 0.0717
  calibration_error: 0.2211

Champion model (by KS): logistic_regression
